**Đọc dataset**

In [1]:
!pip install -q kagglehub[pandas-datasets]

import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd

file_path = "go_emotions_dataset.csv"


df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "shivamb/go-emotions-google-emotions-dataset",
    file_path
)

# Khai báo cấu trúc hệ thống phân loại 28 nhãn chuẩn của Google
emotion_labels = [
    "admiration", "amusement", "anger", "annoyance", "approval", "caring",
    "confusion", "curiosity", "desire", "disappointment", "disapproval",
    "disgust", "embarrassment", "excitement", "fear", "gratitude", "grief",
    "joy", "love", "nervousness", "optimism", "pride", "realization",
    "relief", "remorse", "sadness", "surprise", "neutral"
]



/tmp/ipykernel_5660/2057014050.py:10: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'go-emotions-google-emotions-dataset' dataset.


**Gom cụm**

In [2]:
# Gom cụm các rater đánh giá cùng một câu bình luận lại với nhau
grouped = df.groupby(['id', 'text'])[emotion_labels].sum().reset_index()
print(f"Số lượng câu bình luận độc nhất sau khi gom cụm: {len(grouped)} câu.")

# Nhãn cảm xúc nào được ít nhất 2 chuyên gia chọn mới tính là 1, dưới 2 phiếu bầu sẽ bị hủy (= 0)
for col in emotion_labels:
    grouped[col] = (grouped[col] >= 2).astype(int)

# Tính tổng số nhãn được kích hoạt cho mỗi câu
grouped['has_label'] = grouped[emotion_labels].sum(axis=1)
# Chỉ giữ lại các câu có ít nhất 1 nhãn cảm xúc đạt sự đồng thuận từ 2 người trở lên
grouped_clean = grouped[grouped['has_label'] > 0].drop(columns=['has_label'])

print(f"🔹 Số lượng câu bình luận sạch thu được: {len(grouped_clean)} câu.")

Số lượng câu bình luận độc nhất sau khi gom cụm: 58011 câu.
🔹 Số lượng câu bình luận sạch thu được: 54263 câu.


**Làm sạch**

In [3]:
import re
import unicodedata
import nltk
from nltk.stem import WordNetLemmatizer

# Tải thư viện cần thiết
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

lemmatizer = WordNetLemmatizer()

def clean_text_for_cnn(text, use_lemmatization=False):
    # 1. Tránh lỗi crash nếu dữ liệu bị trống (NaN)
    if not isinstance(text, str):
        return ""

    # 2. Đưa về chữ thường
    text = text.lower()

    # 3. Đồng bộ hóa "dấu nháy" và xử lý Unicode
    text = re.sub(r"[’‘`´]", "'", text)
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')

    # 4. Khai triển từ viết tắt
    contractions = {
        r"i'm": "i am", r"don't": "do not", r"doesn't": "does not",
        r"can't": "cannot", r"won't": "will not", r"isn't": "is not",
        r"aren't": "are not", r"haven't": "have not", r"hasn't": "has not",
        r"didn't": "did not", r"it's": "it is", r"that's": "that is",
        r"you're": "you are", r"i've": "i have", r"i'll": "i will"
    }
    for pattern, replacement in contractions.items():
        text = re.sub(pattern, replacement, text)

    # Vớt những cụm viết tắt đuôi còn sót
    text = re.sub(r"n't", " not", text)
    text = re.sub(r"'re", " are", text)
    text = re.sub(r"'s", " is", text)
    text = re.sub(r"'d", " would", text)
    text = re.sub(r"'ll", " will", text)
    text = re.sub(r"'ve", " have", text)
    text = re.sub(r"'m", " am", text)

    # 5. Lọc ký tự đặc biệt
    text = re.sub(r"[^a-z0-9\s\!\?\_]", " ", text)

    # BỔ SUNG QUAN TRỌNG: Tách dấu ! và ? ra khỏi chữ để Tokenizer đếm đúng
    text = re.sub(r'([!\?])', r' \1 ', text)

    # 6. Ép các ký tự lặp lại vô nghĩa (soooo -> so)
    text = re.sub(r'(.)\1{2,}', r'\1', text)

    # 7. Gom khoảng trắng thừa và cắt tỉa
    text = re.sub(r"\s+", " ", text).strip()

    # 8. Lemmatization (Ép gốc từ)
    if use_lemmatization:
        words = text.split()
        text = " ".join([lemmatizer.lemmatize(w, pos='v') for w in words])

    return text

# ========================================================
# ÁP DỤNG HÀM LÀM SẠCH LÊN TẬP DỮ LIỆU
# ========================================================

df_ready = grouped_clean.copy()
df_ready = df_ready.dropna(subset=['text'])

# FIX: Bật use_lemmatization=True vì lát nữa ta tự train Word2Vec
print("⏳ Đang làm sạch văn bản và ép gốc từ (Lemmatization)...")
df_ready['text'] = df_ready['text'].apply(lambda x: clean_text_for_cnn(x, use_lemmatization=True))

df_ready = df_ready[df_ready['text'] != ""]


⏳ Đang làm sạch văn bản và ép gốc từ (Lemmatization)...


**Chia tập train test valid**

In [4]:
from sklearn.model_selection import train_test_split

# 1. Tách ma trận đầu vào X (văn bản SẠCH) và ma trận đầu ra Y (28 nhãn)
X_text = df_ready['text'].astype(str).values      # <--- Sửa thành df_ready
Y_matrix = df_ready[emotion_labels].values        # <--- Sửa thành df_ready

# 2. Lần tách thứ nhất: Giữ lại 80% cho tập Train, đẩy 20% còn lại vào tập tạm (Temp)
X_train_text, X_temp_text, Y_train, Y_temp = train_test_split(
    X_text, Y_matrix, test_size=0.20, random_state=42
)

# 3. Lần tách thứ hai: Chia đôi 50/50 tập tạm (Temp) để lấy ra đúng 10% Val và 10% Test
X_val_text, X_test_text, Y_val, Y_test = train_test_split(
    X_temp_text, Y_temp, test_size=0.50, random_state=42
)

print(f"Tập Huấn luyện (Train - 80%): {X_train_text.shape[0]} câu.")
print(f"Tập Phát triển/Kiểm định (Val - 10%): {X_val_text.shape[0]} câu.")
print(f"Tập Kiểm thử (Test - 10%): {X_test_text.shape[0]} câu.")
print(f"Tổng cộng: {X_train_text.shape[0] + X_val_text.shape[0] + X_test_text.shape[0]} câu.")

Tập Huấn luyện (Train - 80%): 43408 câu.
Tập Phát triển/Kiểm định (Val - 10%): 5426 câu.
Tập Kiểm thử (Test - 10%): 5426 câu.
Tổng cộng: 54260 câu.


**Tiền trạm**

In [5]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

print("🛠️ ĐANG MÃ HÓA TỪ VỰNG (TOKENIZATION) & ĐỆM CHUỖI (PADDING)...")
print("-" * 60)

# 1. Khởi tạo Tokenizer
# Giới hạn bộ từ vựng ở 20,000 từ phổ biến nhất. Các từ hiếm gặp sẽ bị gộp chung thành nhãn <OOV>
MAX_VOCAB_SIZE = 20000
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")

# BẮT BUỘC: Chỉ fit Tokenizer trên tập Train để chống rò rỉ dữ liệu
print("⏳ Đang Fit Tokenizer trên tập Train...")
tokenizer.fit_on_texts(X_train_text)

# Xem thử có bao nhiêu từ duy nhất trong tập Train
word_index = tokenizer.word_index
print(f"✅ Số lượng từ duy nhất trong từ điển thực tế: {len(word_index)}")

# 2. Chuyển đổi văn bản thành chuỗi các con số
print("⏳ Đang chuyển đổi chữ thành số (texts_to_sequences)...")
X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_val_seq = tokenizer.texts_to_sequences(X_val_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

# 3. Đệm chuỗi (Padding)
# 3. Đệm chuỗi (Padding)
MAX_LEN = 50 # Giới hạn độ dài mỗi câu là 50 từ

print("⏳ Đang ép độ dài câu (Padding)...")
# SỬA Ở ĐÂY: Chuyển 'post' thành 'pre' để bảo toàn masking cho LSTM
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='pre', truncating='pre')
X_val_pad = pad_sequences(X_val_seq, maxlen=MAX_LEN, padding='pre', truncating='pre')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='pre', truncating='pre')

print("\n📊 KIỂM TRA KÍCH THƯỚC MA TRẬN SAU PADDING:")
print("-" * 60)
print(f"   - X_train_pad shape: {X_train_pad.shape}")
print(f"   - X_val_pad shape  : {X_val_pad.shape}")
print(f"   - X_test_pad shape : {X_test_pad.shape}")
print("-" * 60)

🛠️ ĐANG MÃ HÓA TỪ VỰNG (TOKENIZATION) & ĐỆM CHUỖI (PADDING)...
------------------------------------------------------------
⏳ Đang Fit Tokenizer trên tập Train...
✅ Số lượng từ duy nhất trong từ điển thực tế: 21460
⏳ Đang chuyển đổi chữ thành số (texts_to_sequences)...
⏳ Đang ép độ dài câu (Padding)...

📊 KIỂM TRA KÍCH THƯỚC MA TRẬN SAU PADDING:
------------------------------------------------------------
   - X_train_pad shape: (43408, 50)
   - X_val_pad shape  : (5426, 50)
   - X_test_pad shape : (5426, 50)
------------------------------------------------------------


Word2Vec

In [6]:
# 0. Cài đặt thư viện gensim nếu môi trường chưa có
!pip install -q gensim

from gensim.models import Word2Vec
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Chuẩn bị dữ liệu cho Gensim (Đầu vào là list các từ)
sentences_for_w2v = [str(sentence).split() for sentence in X_train_text]

# 2. Huấn luyện mô hình Word2Vec từ đầu
print("⏳ Đang huấn luyện mô hình Word2Vec...")
w2v_model = Word2Vec(sentences=sentences_for_w2v, vector_size=100, window=5, min_count=1, workers=4)

# 3. Đệm chuỗi (Padding)
MAX_LEN = 50 # Giới hạn độ dài mỗi câu là 50 từ

print("⏳ Đang ép độ dài câu (Padding)...")
# SỬA LẠI: Chuyển về 'post' để tương thích với GPU (cuDNN)
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='post', truncating='post')

# 3. Tạo Ma trận Nhúng (Embedding Matrix) tối ưu RAM
EMBEDDING_DIM = 100

# FIX LỖI TỐI ƯU: Chỉ lấy kích thước tối đa là MAX_VOCAB_SIZE (20000)
# Nếu từ điển thực tế nhỏ hơn 20.000 thì lấy len(word_index) + 1
vocab_size = min(len(word_index) + 1, MAX_VOCAB_SIZE)
embedding_matrix = np.zeros((vocab_size, EMBEDDING_DIM))

oov_count = 0
# Bơm vector vào Ma trận
for word, i in word_index.items():
    if i >= vocab_size:
        # Bỏ qua không bơm vector cho những từ nằm ngoài top 20.000 (đã bị Tokenizer vứt bỏ)
        continue

    if word in w2v_model.wv:
        embedding_matrix[i] = w2v_model.wv[word]
    else:
        oov_count += 1

print(f"🔹 Kích thước Embedding Matrix cuối cùng: {embedding_matrix.shape}")
print(f"🔹 Số lượng từ không có vector (OOV) trong top {vocab_size}: {oov_count} từ")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 30.6 MB/s eta 0:00:00
⏳ Đang huấn luyện mô hình Word2Vec...
⏳ Đang ép độ dài câu (Padding)...
🔹 Kích thước Embedding Matrix cuối cùng: (20000, 100)
🔹 Số lượng từ không có vector (OOV) trong top 20000: 16 từ


**Bi LSTM**

In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Bidirectional, Dense, Dropout # Thêm Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryFocalCrossentropy # Thêm Focal Loss

print("🧠 ĐANG XÂY DỰNG KIẾN TRÚC MẠNG BI-LSTM + WORD2VEC...")
print("-" * 60)

# Khởi tạo mô hình
model = Sequential()

# SỬA Ở ĐÂY: Bắt buộc phải có lớp Input để định hình đầu vào và giúp Masking hoạt động
model.add(Input(shape=(MAX_LEN,)))

# 1. Lớp Nhúng (Embedding Layer)
model.add(Embedding(
    input_dim=vocab_size,
    output_dim=EMBEDDING_DIM,
    weights=[embedding_matrix],
    # Đã xóa input_length=MAX_LEN ở đây vì đã dùng lớp Input ở trên
    trainable=True,
    mask_zero=True,
    name="Custom_Word2Vec_Embedding"
))

# 2. Lớp LSTM 2 chiều (Bidirectional LSTM)
model.add(Bidirectional(LSTM(128, return_sequences=False), name="Bi_LSTM_Layer"))

# Dropout 0.3
model.add(Dropout(0.3, name="Dropout_1"))

# 3. Lớp Ẩn (Hidden Dense Layer)
model.add(Dense(64, activation='relu', name="Dense_Feature_Layer"))
model.add(Dropout(0.2, name="Dropout_2"))

# 4. Lớp Đầu ra (Output Layer) cho 28 nhãn cảm xúc
model.add(Dense(28, activation='sigmoid', name="Output_28_Emotions"))

# 5. Biên dịch mô hình (Compile)
optimizer = Adam(learning_rate=0.001)
model.compile(
    # SỬA Ở ĐÂY: Dùng Focal Loss để xử lý mất cân bằng dữ liệu nghiêm trọng
    loss=BinaryFocalCrossentropy(gamma=2.0, alpha=0.25),
    optimizer=optimizer,
    metrics=['binary_accuracy']
)

print("✅ KHỞI TẠO THÀNH CÔNG! Bảng tóm tắt kiến trúc mạng:")
print("=" * 60)
model.summary()

🧠 ĐANG XÂY DỰNG KIẾN TRÚC MẠNG BI-LSTM + WORD2VEC...
------------------------------------------------------------
✅ KHỞI TẠO THÀNH CÔNG! Bảng tóm tắt kiến trúc mạng:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Custom_Word2Vec_Embedding       │ (None, 50, 100)        │     2,000,000 │
│ (Embedding)                     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Bi_LSTM_Layer (Bidirectional)   │ (None, 256)            │       234,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dense_Feature_Layer (Dense)     │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output_28_Emotions (Dense)      │ (None, 28)             │         1,820 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,252,764 (8.59 MB)

 Trainable params: 2,252,764 (8.59 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

print("🚀 BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH BiLSTM + Word2Vec...")
print("-" * 60)

# =====================================================
# CALLBACKS
# =====================================================

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

checkpoint = ModelCheckpoint(
    'best_bilstm_model.keras',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

# =====================================================
# TRAIN
# =====================================================

history = model.fit(
    X_train_pad,
    Y_train,
    validation_data=(X_val_pad, Y_val),
    epochs=20,
    batch_size=128,
    callbacks=[
        early_stop,
        reduce_lr,
        checkpoint
    ],
    verbose=1
)

print("\n✅ HUẤN LUYỆN HOÀN TẤT!")

🚀 BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH BiLSTM + Word2Vec...
------------------------------------------------------------
Epoch 1/20
336/340 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - binary_accuracy: 0.9257 - loss: 0.0597
Epoch 1: val_loss improved from None to 0.03446, saving model to best_bilstm_model.keras

Epoch 1: finished saving model to best_bilstm_model.keras
340/340 ━━━━━━━━━━━━━━━━━━━━ 13s 16ms/step - binary_accuracy: 0.9517 - loss: 0.0457 - val_binary_accuracy: 0.9623 - val_loss: 0.0345 - learning_rate: 0.0010
Epoch 2/20
340/340 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - binary_accuracy: 0.9622 - loss: 0.0355
Epoch 2: val_loss improved from 0.03446 to 0.03064, saving model to best_bilstm_model.keras

Epoch 2: finished saving model to best_bilstm_model.keras
340/340 ━━━━━━━━━━━━━━━━━━━━ 6s 17ms/step - binary_accuracy: 0.9629 - loss: 0.0343 - val_binary_accuracy: 0.9647 - val_loss: 0.0306 - learning_rate: 0.0010
Epoch 3/20
337/340 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - binary_accuracy: 0.9655 - loss: 0.

**Tìm ngưỡng tối ưu F1**

In [9]:
from sklearn.metrics import f1_score
import numpy as np
import time

print("🔎 ĐANG TÌM NGƯỠNG TỐI ƯU (THRESHOLD TUNING) CHO TỪNG NHÃN TRÊN TẬP VAL...")
print("-" * 60)
start_time = time.time()

# 1. Dự đoán xác suất trên tập Validation
print("⏳ Đang tính toán xác suất từ mô hình (model_optimized)...")
# Thêm verbose=0 để ẩn thanh tiến trình của Keras cho đỡ rác màn hình
Y_val_pred_probs = model.predict(X_val_pad, verbose=0)

best_thresholds = {}
f1_scores = {}

# FIX LỖI: Đảm bảo Y_val là Numpy Array để không bị lỗi Y_val[:, i]
Y_val_array = Y_val.values if hasattr(Y_val, 'values') else Y_val

print("🎯 Bắt đầu quét ngưỡng cho 28 nhãn cảm xúc...\n")
for i, label in enumerate(emotion_labels):
    best_thresh = 0.5
    best_f1 = 0.0

    # Dùng bước nhảy 0.05 (đủ chuẩn xác và nhanh hơn rất nhiều so với 0.01)
    for thresh in np.arange(0.05, 0.96, 0.01):
        thresh = round(thresh, 2) # Xử lý lỗi số thực của Numpy (vd: 0.15000000002)

        y_pred_binary = (Y_val_pred_probs[:, i] >= thresh).astype(int)
        y_true = Y_val_array[:, i]

        current_f1 = f1_score(y_true, y_pred_binary, zero_division=0)

        if current_f1 > best_f1:
            best_f1 = current_f1
            best_thresh = thresh

    best_thresholds[label] = best_thresh
    f1_scores[label] = best_f1

    # In ra với định dạng căn lề chuẩn
    print(f"Nhãn: [{label.upper().ljust(15)}] | Ngưỡng tối ưu: {best_thresh:.2f} | F1-Score: {best_f1:.4f}")

macro_f1 = np.mean(list(f1_scores.values()))
end_time = time.time()

print("=" * 60)
print(f"🏆 ĐIỂM MACRO F1 TRUNG BÌNH CỦA MÔ HÌNH (VAL): {macro_f1:.4f}")
print(f"⏳ Thời gian quét: {end_time - start_time:.2f} giây")
print("=" * 60)

🔎 ĐANG TÌM NGƯỠNG TỐI ƯU (THRESHOLD TUNING) CHO TỪNG NHÃN TRÊN TẬP VAL...
------------------------------------------------------------
⏳ Đang tính toán xác suất từ mô hình (model_optimized)...
🎯 Bắt đầu quét ngưỡng cho 28 nhãn cảm xúc...

Nhãn: [ADMIRATION     ] | Ngưỡng tối ưu: 0.46 | F1-Score: 0.6536
Nhãn: [AMUSEMENT      ] | Ngưỡng tối ưu: 0.48 | F1-Score: 0.7665
Nhãn: [ANGER          ] | Ngưỡng tối ưu: 0.46 | F1-Score: 0.4025
Nhãn: [ANNOYANCE      ] | Ngưỡng tối ưu: 0.35 | F1-Score: 0.3162
Nhãn: [APPROVAL       ] | Ngưỡng tối ưu: 0.39 | F1-Score: 0.3082
Nhãn: [CARING         ] | Ngưỡng tối ưu: 0.33 | F1-Score: 0.2929
Nhãn: [CONFUSION      ] | Ngưỡng tối ưu: 0.33 | F1-Score: 0.2857
Nhãn: [CURIOSITY      ] | Ngưỡng tối ưu: 0.35 | F1-Score: 0.4066
Nhãn: [DESIRE         ] | Ngưỡng tối ưu: 0.43 | F1-Score: 0.4255
Nhãn: [DISAPPOINTMENT ] | Ngưỡng tối ưu: 0.29 | F1-Score: 0.1718
Nhãn: [DISAPPROVAL    ] | Ngưỡng tối ưu: 0.32 | F1-Score: 0.2879
Nhãn: [DISGUST        ] | Ngưỡng tối ưu: 0.34 

Gradio

In [10]:
!pip install -q gradio

import gradio as gr
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

def predict_emotion(text):
    if not text.strip():
        return "Vui lòng nhập văn bản!"

    # 1. Làm sạch văn bản (sử dụng hàm clean_text_for_cnn đã định nghĩa ở trên)
    cleaned_text = clean_text_for_cnn(text, use_lemmatization=True)

    # 2. Chuyển đổi sang chuỗi số và Padding
    seq = tokenizer.texts_to_sequences([cleaned_text])
    padded = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')

    # 3. Dự đoán xác suất từ mô hình Bi-LSTM
    probas = model.predict(padded, verbose=0)[0]

    # 4. Trình bày kết quả dựa trên bộ ngưỡng best_thresholds đã tìm được
    results = {}
    activated = False

    for i, label in enumerate(emotion_labels):
        prob = probas[i]
        threshold_val = best_thresholds[label]

        if prob >= threshold_val:
            results[label] = float(prob)
            activated = True

    # Nếu không nhãn nào vượt ngưỡng, lấy nhãn có xác suất cao nhất
    if not activated:
        top_idx = np.argmax(probas)
        results[emotion_labels[top_idx]] = float(probas[top_idx])

    return results

# Khởi tạo giao diện Gradio
iface = gr.Interface(
    fn=predict_emotion,
    inputs=gr.Textbox(lines=2, placeholder="Nhập câu tiếng Anh tại đây...", label="Input Text"),
    outputs=gr.Label(num_top_classes=5, label="Phân loại cảm xúc"),
    title="🚀 Google GoEmotions Predictor (Bi-LSTM)",
    description="Ứng dụng nhận diện 28 sắc thái cảm xúc dựa trên Bi-LSTM + Word2Vec và bộ ngưỡng tối ưu.",
    examples=[
        ["I am so excited!"],
        ["This is disappointing."],
        ["Thank you for your help."],
        ["Wow, I am so surprised and happy for you!"],
        ["I love this, it is absolutely beautiful and amazing!"]
    ]
)

# Khởi chạy giao diện
iface.launch(share=True, inline=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://867f5d5aee2ae4020f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
